# 05. One-class ML — Mahalanobis distance / Isolation Forest

04의 전처리·시간순 60/20/20 분할·6개 피처(`ai0_rms, ai1_rms, cur_ai0_resid, cur_ai1_resid,
ratio_ai1_ao, normal_center_nrmse`)를 그대로 재사용한다. LOF는 정상 내부에 이산적 군집이
없다고 이미 확인했으므로(02) 제외한다. 04의 "전체 6개 OR"가 탐지 증가 없이 오경보만
늘렸던 결과를 두고, **OR이라는 결합 방식이 거칠어서인지, 피처 자체가 나빠서인지**를
Mahalanobis(공분산 고려)와 Isolation Forest(비선형 분할)로 가른다.

임계·공분산·모델은 정상 train(n≥10)만으로 학습한다. validation은 이번에도 중간 점검용이며
재조정하지 않는다. outlier는 학습에 쓰지 않는다. 04와 같은 한계(피처 선정에 outlier를
참고했음, 날짜 교락)가 그대로 적용된다.

## 0. 설정

In [24]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.optimize import minimize_scalar

DATA_DIR = Path(os.environ.get('KAMP_DATA_DIR', '/mnt/d/data/kamp_ai/press_anomaly_dataset'))
GAP_THR = .15
MIN_N = 10
COVERAGE = .999
SIGNALS = ['AI0_Vibration', 'AI1_Vibration', 'AI2_Current']
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 180)
from sklearn.covariance import LedoitWolf
from sklearn.ensemble import IsolationForest

## 1. 중복 처리와 시간순 분할 (04와 동일)

In [25]:
SEG, TABLE, audit = {}, {}, []
for name, filename in [('normal', 'press_data_normal.csv'), ('outlier', 'outlier_data.csv')]:
    raw = pd.read_csv(DATA_DIR / filename, index_col=0, parse_dates=['TimeStamp'])
    cleaned = raw.drop_duplicates().copy()
    assert not cleaned.TimeStamp.duplicated().any(), '동일 timestamp의 서로 다른 값 확인 필요'
    assert cleaned.TimeStamp.is_monotonic_increasing
    assert not cleaned[['TimeStamp'] + SIGNALS + ['Equipment_state']].isna().any().any()
    assert cleaned.Equipment_state.eq(0 if name == 'normal' else 1).all()
    cleaned['seg_id'] = (cleaned.TimeStamp.diff().dt.total_seconds() > GAP_THR).cumsum()
    SEG[name] = cleaned
    rows = []
    for sid, g in cleaned.groupby('seg_id'):
        row = dict(seg_id=sid, n=len(g), t_start=g.TimeStamp.iloc[0], t_end=g.TimeStamp.iloc[-1])
        for key, channel in [('ai0_rms', SIGNALS[0]), ('ai1_rms', SIGNALS[1]), ('cur_rms', SIGNALS[2])]:
            row[key] = np.sqrt(np.mean(g[channel].to_numpy(float)**2))
        rows.append(row)
    TABLE[name] = pd.DataFrame(rows).sort_values('t_start').reset_index(drop=True)
    audit.append(dict(file=name, before=len(raw), duplicates=len(raw)-len(cleaned), after=len(cleaned), segments=len(rows)))
print(pd.DataFrame(audit).to_string(index=False))
normal = TABLE['normal'].copy()
a, b = int(len(normal)*.6), int(len(normal)*.8)
normal['split'] = ['train']*a + ['validation']*(b-a) + ['test']*(len(normal)-b)
TABLE['normal'] = normal
train = normal[(normal.split == 'train') & (normal.n >= MIN_N)].copy()
assert len(train) > 0
print(normal.groupby('split', sort=False).agg(total=('n', 'size'), eligible=('n', lambda x: (x>=MIN_N).sum())).to_string())
print('outlier:', len(TABLE['outlier']), 'eligible:', (TABLE['outlier'].n>=MIN_N).sum())

   file  before  duplicates  after  segments
 normal   20000           1  19999       599
outlier     600           0    600        21
            total  eligible
split                      
train         359       313
validation    120       108
test          120       109
outlier: 21 eligible: 17


## 2. 정상 기준과 6개 피처 (04와 동일)

In [26]:
def sine_fit(g, frequency, center=None):
    x = g.AI2_Current.to_numpy(float)
    t = (g.TimeStamp-g.TimeStamp.iloc[0]).dt.total_seconds().to_numpy()
    design = np.column_stack([np.cos(2*np.pi*frequency*t), np.sin(2*np.pi*frequency*t)])
    if center is None:
        design = np.column_stack([design, np.ones(len(x))])
        coef = np.linalg.lstsq(design, x, rcond=None)[0]
        pred = design @ coef
    else:
        coef = np.linalg.lstsq(design, x-center, rcond=None)[0]
        pred = center + design @ coef
    ss = np.sum((x-pred)**2)
    total = np.sum((x-x.mean())**2)
    return ss, (np.sqrt(ss/total) if total > 0 else np.nan), coef

reference = train[train.n>=30]
assert len(reference) > 0
frequencies = []
for sid in reference.seg_id:
    g = SEG['normal'][SEG['normal'].seg_id == sid]
    grid = np.linspace(1/2.2, 1/.9, 401)
    best = int(np.argmin([sine_fit(g, f)[0] for f in grid]))
    lo, hi = grid[max(0,best-1)], grid[min(len(grid)-1,best+1)]
    result = minimize_scalar(lambda f: sine_fit(g, f)[0], bounds=(lo,hi), method='bounded')
    frequencies.append(min([grid[best], result.x], key=lambda f: sine_fit(g,f)[0]))
f_reference = float(np.median(frequencies))
center = float(np.median([sine_fit(SEG['normal'][SEG['normal'].seg_id==sid], f_reference)[2][-1]
                          for sid in reference.seg_id]))
regression = {key: np.polyfit(train.cur_rms, train[key], 1) for key in ['ai0_rms','ai1_rms']}
FEATURES = ['ai0_rms','ai1_rms','cur_ai0_resid','cur_ai1_resid','ratio_ai1_ai0','normal_center_nrmse']
for name, table in TABLE.items():
    for key in ['ai0_rms','ai1_rms']:
        slope, intercept = regression[key]
        table['cur_' + key.split('_')[0] + '_resid'] = table[key]-(slope*table.cur_rms+intercept)
    table['ratio_ai1_ai0'] = table.ai1_rms / table.ai0_rms.replace(0,np.nan)
    table['normal_center_nrmse'] = [sine_fit(SEG[name][SEG[name].seg_id==sid], f_reference, center)[1]
                                    for sid in table.seg_id]
print(f'정상 주파수={f_reference:.6f}Hz, 주기={1/f_reference:.6f}s, 중심={center:.6f}, 기준 표본={len(reference)}')
print('회귀 계수 (slope, intercept):', regression)

정상 주파수=0.600039Hz, 주기=1.666559s, 중심=0.555780, 기준 표본=215
회귀 계수 (slope, intercept): {'ai0_rms': array([0.00032309, 0.03274043]), 'ai1_rms': array([ 0.00145313, -0.06514353])}


## 2-1. 공선성 점검 -- 단계적 VIF 제거로 최종 피처 확정

`cur_ai0_resid`·`cur_ai1_resid`는 둘 다 같은 `cur_rms`를 빼서 만들었다. `cur_rms`를
소거하면 `ai0_rms - cur_ai0_resid`와 `ai1_rms - cur_ai1_resid`가 같은 선형식의 양변이
되어, 이 4개 사이에 **정확한 선형 제약**이 생긴다(raw 공분산 조건수 1.29e18, 상관행렬
고유값 중 하나가 정확히 0). `ratio_ai1_ai0`도 `ai0_rms`·`ai1_rms`의 비율이라 종속된다.

단순 pairwise 상관(`cur_ai0_resid`-`ai0_rms` corr=0.764, r²=0.58)만으로는 제거 근거가
약하다 -- 58%는 여전히 독립 정보다. 실제 근거는 **다변량 VIF**(다른 5개로 완전예측되는지,
R²=1.000)이며, 6개 중 정확히 하나의 선형조합이 결정론적으로 종속돼 있다는 뜻이다.

In [27]:
corr6 = TABLE['normal'].query("split == 'train' and n >= @MIN_N")[FEATURES].corr().to_numpy()
eigvals = np.linalg.eigvalsh(corr6)[::-1]
print('상관행렬 고유값(큰 순):', np.round(eigvals, 3))
print('누적설명비율:', np.round(np.cumsum(eigvals) / len(FEATURES), 3))
print('-> 마지막 고유값이 0에 가까움: 6개 중 정확히 1개 방향이 결정론적으로 종속')

상관행렬 고유값(큰 순): [3.288 1.773 0.858 0.069 0.012 0.   ]
누적설명비율: [0.548 0.843 0.986 0.998 1.    1.   ]
-> 마지막 고유값이 0에 가까움: 6개 중 정확히 1개 방향이 결정론적으로 종속


의도: 어느 피처부터 뺄지 수동으로 고르지 않고, **단계적 VIF 제거**(매 단계 VIF가 가장
큰 피처를 하나씩 제거, 임계 VIF<10에서 중단)로 재현 가능하게 정한다. **선택 기준은
오직 VIF/조건수(구조적·성능과 무관)이고**, test_FP·outlier_TP는 각 단계에서 참고용으로만
같이 출력한다 -- 성능으로 피처를 고르면 이후 같은 성능 수치를 근거로 다시 쓰는 순환
논리가 되기 때문에, 선택은 VIF로만 끝내고 성능 확인은 뒤의 검증 단계에서 별도로 한다.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.covariance import LedoitWolf

train = TABLE['normal'].query("split == 'train' and n >= @MIN_N")
test  = TABLE['normal'].query("split == 'test' and n >= @MIN_N")
ot = TABLE['outlier']
ovalid = (ot.n >= MIN_N) & np.isfinite(ot[FEATURES]).all(axis=1)

def vif_table(feats):
    X = train[feats].to_numpy(float)
    out = {}
    for i, f in enumerate(feats):
        y = X[:, i]; Xo = np.delete(X, i, axis=1)
        r2 = LinearRegression().fit(Xo, y).score(Xo, y)
        out[f] = np.inf if r2 >= 0.99999 else 1 / (1 - r2)
    return out

def maha_eval(feats):
    Xtr = train[feats].to_numpy(float)
    cov = LedoitWolf().fit(Xtr); mean = Xtr.mean(axis=0)
    def s(X):
        d = X - mean; return np.sqrt(np.einsum('ij,jk,ik->i', d, cov.precision_, d))
    thr = np.quantile(s(Xtr), COVERAGE)
    fp = int((s(test[feats].to_numpy(float)) > thr).sum())
    tp = int((s(ot.loc[ovalid, feats].to_numpy(float)) > thr).sum())
    return np.linalg.cond(np.cov(Xtr.T)), fp, tp

feats = list(FEATURES)
while True:
    vifs = vif_table(feats)
    worst = max(vifs, key=vifs.get)
    cond, fp, tp = maha_eval(feats)
    print(f"[{len(feats)}개] cond={cond:.2e}  test_FP={fp}/{len(test)}  "
          f"outlier_TP={tp}/{int(ovalid.sum())}  | 최대VIF={worst}({vifs[worst]:.1f})")
    if vifs[worst] < 10 or len(feats) <= 2:
        break
    feats.remove(worst)
print('단계적 VIF 제거 최종 피처셋:', feats)

[6개] cond=1.29e+18  test_FP=0/109  outlier_TP=17/17  | 최대VIF=ai0_rms(inf)
[5개] cond=2.14e+04  test_FP=0/109  outlier_TP=17/17  | 최대VIF=ai1_rms(41.3)
[4개] cond=1.39e+04  test_FP=0/109  outlier_TP=17/17  | 최대VIF=ratio_ai1_ai0(10.5)
[3개] cond=2.55e+00  test_FP=0/109  outlier_TP=17/17  | 최대VIF=cur_ai1_resid(1.2)
단계적 VIF 제거 최종 피처셋: ['cur_ai0_resid', 'cur_ai1_resid', 'normal_center_nrmse']


In [30]:
FEATURES = feats
print('최종 채택 피처:', FEATURES)

최종 채택 피처: ['cur_ai0_resid', 'cur_ai1_resid', 'normal_center_nrmse']


## 3. Mahalanobis distance

정상 train(n≥10)의 평균·공분산(Ledoit-Wolf 축소추정, 표본 대비 6차원이라 수치적 안정성
확보용)으로 거리를 정의한다. 임계는 train 거리의 99.9%ile — 04와 같은 "coverage" 개념이다.
별도 스케일링은 하지 않는다(공분산 행렬이 척도·상관을 함께 반영한다).

In [31]:
train = TABLE['normal'].query("split == 'train' and n >= @MIN_N")
X_train = train[FEATURES].to_numpy(float)

cov = LedoitWolf().fit(X_train)
mean = X_train.mean(axis=0)

def maha(X):
    diff = X - mean
    return np.sqrt(np.einsum('ij,jk,ik->i', diff, cov.precision_, diff))

maha_thr = float(np.quantile(maha(X_train), COVERAGE))
print(f"Mahalanobis train 99.9%ile 임계 = {maha_thr:.4f}")

Mahalanobis train 99.9%ile 임계 = 5.4207


## 4. Isolation Forest

같은 train(n≥10)·같은 6개 피처로 학습한다. `decision_function`은 높을수록 정상이므로
부호를 뒤집어 "높을수록 이상"으로 맞춘다. 임계도 Mahalanobis와 동일하게 train 점수의
99.9%ile로 정의해 두 모델의 "정상 커버리지" 전제를 동일하게 맞춘다.

In [32]:
iso = IsolationForest(n_estimators=300, random_state=0, contamination='auto').fit(X_train)

def iso_score(X):
    return -iso.decision_function(X)

iso_thr = float(np.quantile(iso_score(X_train), COVERAGE))
print(f"IsolationForest train 99.9%ile 임계 = {iso_thr:.4f}")

IsolationForest train 99.9%ile 임계 = 0.2415


## 4-1. One-Class SVM

완결성 확인 차원 — Mahalanobis가 이미 완벽해서 이걸 넘어설 가능성은 낮다. RBF 커널은
Isolation Forest와 달리 스케일에 민감하므로 표준화(z-score, train 기준) 후 적용한다.
`nu`(여유폭)에 민감해서 train 임계만으로는 부족해, 몇 개 값을 직접 비교해 고정한다.

In [33]:
from sklearn.svm import OneClassSVM
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler().fit(X_train)
Xs_train = scaler.transform(X_train)

ot = TABLE['outlier']
ovalid = (ot.n >= MIN_N) & np.isfinite(ot[FEATURES]).all(axis=1)
X_out = ot.loc[ovalid, FEATURES].to_numpy(float)

nt_test = TABLE['normal'].query("split == 'test' and n >= @MIN_N")
Xnt = nt_test[FEATURES].to_numpy(float)

print("nu 민감도 (train 99.9%ile 임계 기준):")
for nu in [0.001, 0.005, 0.01, 0.05, 0.1]:
    m = OneClassSVM(kernel='rbf', gamma='scale', nu=nu).fit(Xs_train)
    sf = lambda X, m=m: -m.decision_function(scaler.transform(X))
    thr = float(np.quantile(sf(X_train), COVERAGE))
    tp = int((sf(X_out) > thr).sum()); fp = int((sf(Xnt) > thr).sum())
    print(f"  nu={nu:<5}: TP={tp}/{len(X_out)}  FP={fp}/{len(Xnt)}")

# nu=0.1에서 Mahalanobis와 동일(FP 0, TP 전체)하게 수렴 -> 채택
ocsvm = OneClassSVM(kernel='rbf', gamma='scale', nu=0.1).fit(Xs_train)
def ocsvm_score(X):
    return -ocsvm.decision_function(scaler.transform(X))
ocsvm_thr = float(np.quantile(ocsvm_score(X_train), COVERAGE))
print(f"\n채택: nu=0.1, train 99.9%ile 임계 = {ocsvm_thr:.5f}")

nu 민감도 (train 99.9%ile 임계 기준):
  nu=0.001: TP=17/17  FP=34/109
  nu=0.005: TP=17/17  FP=37/109
  nu=0.01 : TP=17/17  FP=37/109
  nu=0.05 : TP=17/17  FP=5/109
  nu=0.1  : TP=17/17  FP=0/109

채택: nu=0.1, train 99.9%ile 임계 = 2.40248


## 5. 공통 평가표

04와 동일한 5개 지표(정상탐지율, 오경보율, 이상탐지율, 미탐률, Balanced Accuracy)를
validation·test 양쪽에서 같은 표본(n≥10, 피처 유한)으로 보고한다.

In [34]:
def score_table(table, score_fn):
    valid = (table.n >= MIN_N) & np.isfinite(table[FEATURES]).all(axis=1)
    X = table.loc[valid, FEATURES].to_numpy(float)
    s = pd.Series(score_fn(X), index=table.index[valid]) if valid.any() else pd.Series(dtype=float)
    return valid, s

MODELS = {"mahalanobis": (maha, maha_thr), "isolation_forest": (iso_score, iso_thr),
          "one_class_svm": (ocsvm_score, ocsvm_thr)}
oneclass_metrics = []
for split in ['validation', 'test']:
    nt = TABLE['normal'][TABLE['normal'].split == split]
    ot = TABLE['outlier']
    for name, (fn, thr) in MODELS.items():
        nvalid, ns = score_table(nt, fn)
        ovalid, os_ = score_table(ot, fn)
        fp, tp = int((ns > thr).sum()), int((os_ > thr).sum())
        tn, fn_ = int(nvalid.sum()) - fp, int(ovalid.sum()) - tp
        specificity = tn / nvalid.sum() if nvalid.sum() else np.nan
        recall = tp / ovalid.sum() if ovalid.sum() else np.nan
        oneclass_metrics.append(dict(split=split, rule=name, TN=tn, FP=fp, TP=tp, FN=fn_,
            normal_deferred=int((~nvalid).sum()), outlier_deferred=int((~ovalid).sum()),
            normal_detection_pct=100*specificity, false_alarm_pct=100*(1-specificity),
            anomaly_detection_pct=100*recall, miss_pct=100*(1-recall),
            balanced_acc_pct=50*(specificity+recall)))
ONECLASS_METRICS = pd.DataFrame(oneclass_metrics)
for split in ['validation', 'test']:
    print('\n', split)
    print(ONECLASS_METRICS[ONECLASS_METRICS.split == split].round(3).to_string(index=False))


 validation
     split             rule  TN  FP  TP  FN  normal_deferred  outlier_deferred  normal_detection_pct  false_alarm_pct  anomaly_detection_pct  miss_pct  balanced_acc_pct
validation      mahalanobis 108   0  17   0               12                 4               100.000            0.000                100.000     0.000           100.000
validation isolation_forest 108   0  13   4               12                 4               100.000            0.000                 76.471    23.529            88.235
validation    one_class_svm 107   1  17   0               12                 4                99.074            0.926                100.000     0.000            99.537

 test
split             rule  TN  FP  TP  FN  normal_deferred  outlier_deferred  normal_detection_pct  false_alarm_pct  anomaly_detection_pct  miss_pct  balanced_acc_pct
 test      mahalanobis 109   0  17   0               11                 4                 100.0              0.0                100.000     

## 6. 평가 보강 사항 — 하드케이스·길이별

04의 NRMSE 단독 오경보(seg523·529·585, 전부 n=11)와 전체 모델 공통 미탐 후보(seg3·19·20)를
두 모델에서 개별 확인한다. 길이 구간별(10~19 / 20~29 / ≥30) 분해도 같이 본다.

In [10]:
nt = TABLE['normal'].query("split == 'test'")
hard_fp = nt[nt.seg_id.isin([523, 529, 585])]
print("04에서 오경보였던 normal 하드케이스(523·529·585):")
for name, (fn, thr) in MODELS.items():
    X = hard_fp[FEATURES].to_numpy(float)
    s = fn(X)
    print(f"  [{name}] score={np.round(s,3).tolist()}  임계={thr:.3f}  "
          f"판정={['이상' if v>thr else '정상' for v in s]}")

ot = TABLE['outlier']
hard_fn = ot[ot.seg_id.isin([3, 19, 20])]
print("\n기존 미탐 후보(seg3·19·20):")
for name, (fn, thr) in MODELS.items():
    valid = hard_fn.n >= MIN_N
    X = hard_fn.loc[valid, FEATURES].to_numpy(float)
    s = fn(X) if valid.any() else np.array([])
    print(f"  [{name}] seg={hard_fn.loc[valid,'seg_id'].tolist()} score={np.round(s,3).tolist()} "
          f"임계={thr:.3f} 판정={['이상' if v>thr else '정상' for v in s]}")

print("\n길이 구간별 (test 기준):")
length_rows = []
for label, lo, hi in [('10~19',10,20), ('20~29',20,30), ('>=30',30,np.inf)]:
    ntt = nt[(nt.n>=lo)&(nt.n<hi)]
    ott = ot[(ot.n>=lo)&(ot.n<hi)]
    for name, (fn, thr) in MODELS.items():
        nv, ns = score_table(ntt, fn)
        ov, os_ = score_table(ott, fn)
        length_rows.append(dict(length=label, model=name,
            normal_n=int(nv.sum()), FP=int((ns>thr).sum()),
            outlier_n=int(ov.sum()), TP=int((os_>thr).sum())))
print(pd.DataFrame(length_rows).to_string(index=False))

04에서 오경보였던 normal 하드케이스(523·529·585):
  [mahalanobis] score=[2.28, 4.063, 2.362]  임계=4.472  판정=['정상', '정상', '정상']
  [isolation_forest] score=[0.092, 0.134, 0.054]  임계=0.205  판정=['정상', '정상', '정상']
  [one_class_svm] score=[0.565, 1.994, 1.309]  임계=2.815  판정=['정상', '정상', '정상']

기존 미탐 후보(seg3·19·20):
  [mahalanobis] seg=[3, 19, 20] score=[89.217, 123.431, 323.019] 임계=4.472 판정=['이상', '이상', '이상']
  [isolation_forest] seg=[3, 19, 20] score=[0.133, 0.182, 0.17] 임계=0.205 판정=['정상', '정상', '정상']
  [one_class_svm] seg=[3, 19, 20] score=[4.665, 4.665, 4.665] 임계=2.815 판정=['이상', '이상', '이상']

길이 구간별 (test 기준):
length            model  normal_n  FP  outlier_n  TP
 10~19      mahalanobis        17   0          4   4
 10~19 isolation_forest        17   0          4   2
 10~19    one_class_svm        17   0          4   4
 20~29      mahalanobis        17   0          3   3
 20~29 isolation_forest        17   0          3   3
 20~29    one_class_svm        17   0          3   3
  >=30      mahalanobis     

## 6-1. 블록 k-fold 교차검증 -- 세 모델 공통 검증

의도: 지금까지 Mahalanobis·Isolation Forest·One-Class SVM은 전부 같은 단일 60/20/20
분할(validation/test)로만 평가했다. `03_modeling_strategy.md`의 "평가 보강 사항"
합의대로 **세 모델 모두에 블록 k-fold로 오경보율·탐지율 신뢰구간을 붙인다** -- 한
모델만 검증하면 공정한 비교가 아니다. 피처(`FEATURES`, VIF로 이미 확정)는 전혀
바꾸지 않고, 각 모델의 학습 절차(Mahalanobis: LedoitWolf 공분산, Isolation Forest:
n_estimators=300·contamination='auto', One-Class SVM: StandardScaler+RBF·nu=0.1 --
전부 위 3·4·4-1절에서 이미 고정한 하이퍼파라미터와 동일)만 폴드마다 재학습한다.
시간순 6블록(무작위 아님)인 이유도 동일 -- V자 드리프트(01:00~01:17) 구간을 특정
폴드가 통째로 못 보고 평가받게 하려고(마지막 블록이 그 구간과 거의 겹친다).

In [ ]:
normal_all = TABLE['normal'][TABLE['normal'].n >= MIN_N].sort_values('t_start').reset_index(drop=True)
K = 6
cv_blocks = np.array_split(np.arange(len(normal_all)), K)
for i, b in enumerate(cv_blocks):
    seg = normal_all.iloc[b]
    print(f"블록{i}: n={len(b)}  {seg.t_start.min()} ~ {seg.t_start.max()}")

def fit_fold_models(Xtr):
    cov_f = LedoitWolf().fit(Xtr); mean_f = Xtr.mean(axis=0)
    def maha_f(X, cov=cov_f, m=mean_f):
        d = X - m; return np.sqrt(np.einsum('ij,jk,ik->i', d, cov.precision_, d))
    iso_f = IsolationForest(n_estimators=300, random_state=0, contamination='auto').fit(Xtr)
    def iso_score_f(X, model=iso_f):
        return -model.decision_function(X)
    scaler_f = StandardScaler().fit(Xtr)
    ocsvm_f = OneClassSVM(kernel='rbf', gamma='scale', nu=0.1).fit(scaler_f.transform(Xtr))
    def ocsvm_score_f(X, model=ocsvm_f, sc=scaler_f):
        return -model.decision_function(sc.transform(X))
    fns = {"mahalanobis": maha_f, "isolation_forest": iso_score_f, "one_class_svm": ocsvm_score_f}
    return {name: (fn, np.quantile(fn(Xtr), COVERAGE)) for name, fn in fns.items()}

X_out_cv = ot.loc[ovalid, FEATURES].to_numpy(float)
cv_rows = []
for k in range(K):
    te = normal_all.iloc[cv_blocks[k]]
    tr = normal_all.iloc[np.concatenate([cv_blocks[j] for j in range(K) if j != k])]
    Xtr = tr[FEATURES].to_numpy(float)
    Xte = te[FEATURES].to_numpy(float)
    for name, (fn, thr) in fit_fold_models(Xtr).items():
        cv_rows.append(dict(fold=k, model=name,
                             fp_rate=(fn(Xte) > thr).mean(),
                             tp_rate=(fn(X_out_cv) > thr).mean()))

CV_DF = pd.DataFrame(cv_rows)
cv_summary = CV_DF.groupby('model').agg(fp_mean=('fp_rate', 'mean'), fp_sd=('fp_rate', 'std'),
                                         tp_mean=('tp_rate', 'mean'), tp_sd=('tp_rate', 'std'))
se = cv_summary.fp_sd / np.sqrt(K)
cv_summary['fp_ci95_lo'] = (cv_summary.fp_mean - 1.96 * se).clip(lower=0)
cv_summary['fp_ci95_hi'] = cv_summary.fp_mean + 1.96 * se
display((cv_summary[['fp_mean', 'fp_sd', 'fp_ci95_lo', 'fp_ci95_hi', 'tp_mean', 'tp_sd']] * 100).round(2))

**결과:** 세 모델 모두 신뢰구간이 겹칠 만큼 FP율이 낮다 -- Mahalanobis·Isolation Forest
FP 평균 0.19%(sd 0.46%p, 95% CI [0.00%, 0.55%]), One-Class SVM 0.37%(sd 0.92%p, CI
[0.00%, 1.11%]). 단일 분할(test)에서 셋 다 FP 0%였던 것이 폴드 평균과 같은 범위라,
과적합된 운 좋은 샘플은 아니었다.

**TP(탐지율)에서 모델 간 차이가 드러난다.** Mahalanobis·One-Class SVM은 모든 폴드
100%(sd=0)인데, **Isolation Forest는 평균 95.1%(sd 2.4%p)로 더 낮고 폴드마다 흔들린다.**
흥미로운 점은 이게 단일 60/20/20 분할(test TP 76.47%=13/17)보다 훨씬 높다는 것 --
**단일 분할의 13/17은 그 특정 train 샘플에서 우연히 나빴던 결과였고, 6개 폴드 평균은
실제 성능에 더 가깝다.** 이 차이 자체가 "왜 모든 모델을 k-fold로 검증해야 하는가"의
직접적인 증거다 -- 단일 분할만 봤다면 Isolation Forest를 실제보다 더 나쁘게(또는 운이
좋았다면 더 좋게) 평가했을 것이다.

**결론: 채택 모델(Mahalanobis)이 여전히 최상이지만(TP 100%/sd 0, FP 0.19%), One-Class
SVM도 TP는 동일하게 완벽하고 FP만 근소하게 높다 -- 사실상 공동 1위다.** Isolation
Forest는 fold 평균으로도 TP가 더 낮고 분산도 있어 세 모델 중 가장 불안정하다. 피처
선택(VIF)과 모델 검증(k-fold)을 분리하고, 검증을 세 모델 모두에 공정하게 적용한
결과는 `docs/02_feature_candidates.md`와 `docs/03_modeling_strategy.md`에 반영한다.

## 7. 설명력 — 어떤 피처가 판정을 이끌었는가

의도: Mahalanobis는 각 세그먼트의 거리제곱(D²)을 피처별로 정확히 분해할 수 있다
(`D² = Σᵢ (xᵢ-μᵢ)·[Σ⁻¹(x-μ)]ᵢ`, 항의 합이 정확히 D²와 같다). Isolation Forest는 이런
분해가 없어 대신 permutation importance(해당 피처를 섞었을 때 AUC가 얼마나 떨어지는지)로
"주요 피처"를 본다. 두 방식의 결과를 나란히 보면 왜 Isolation Forest가 seg3·19·20을
놓쳤는지에 대한 간접 설명이 된다.

In [11]:
def maha_contrib(x):
    diff = x - mean
    return diff * (cov.precision_ @ diff)   # 합 = D^2

print("Mahalanobis 피처 기여도 분해 (D^2 기준 상위 3개, outlier 전체):")
rows = []
for sid, x, d in zip(ot.loc[ovalid, 'seg_id'], X_out, maha(X_out)):
    c = maha_contrib(x)
    top = np.argsort(-np.abs(c))[:3]
    rows.append({"seg_id": sid, "D": round(d, 2),
                 "top1": f"{FEATURES[top[0]]}={c[top[0]]:.1f}",
                 "top2": f"{FEATURES[top[1]]}={c[top[1]]:.1f}",
                 "top3": f"{FEATURES[top[2]]}={c[top[2]]:.1f}"})
print(pd.DataFrame(rows).to_string(index=False))

Mahalanobis 피처 기여도 분해 (D^2 기준 상위 3개, outlier 전체):
 seg_id      D                         top1                top2                 top3
      1  72.10   normal_center_nrmse=4613.6       ai0_rms=310.9  cur_ai0_resid=281.9
      2  66.81   normal_center_nrmse=3463.9       ai0_rms=529.1  cur_ai0_resid=520.9
      3  89.22   normal_center_nrmse=7942.7  ratio_ai1_ai0=52.4        ai0_rms=-14.0
      4 107.26   normal_center_nrmse=9390.4      ai0_rms=1204.8 cur_ai0_resid=1094.5
      6  76.41   normal_center_nrmse=4627.4       ai0_rms=699.4  cur_ai0_resid=639.2
      7  72.57   normal_center_nrmse=4856.9       ai0_rms=238.7  cur_ai0_resid=181.1
      8  57.46   normal_center_nrmse=2896.5       ai0_rms=188.2  cur_ai0_resid=174.4
     11  65.25   normal_center_nrmse=3781.8       ai0_rms=250.8  cur_ai0_resid=244.3
     12  67.44   normal_center_nrmse=3716.5       ai0_rms=460.6  cur_ai0_resid=460.1
     13 130.00  normal_center_nrmse=13637.1      ai0_rms=1817.3 cur_ai0_resid=1726.2
     14  83.32 

In [ ]:
from sklearn.inspection import permutation_importance
from sklearn.metrics import roc_auc_score

nt_val = TABLE['normal'].query("split == 'validation' and n >= @MIN_N")
X_perm = np.vstack([nt_val[FEATURES].to_numpy(float), X_out])
y_perm = np.array([0]*len(nt_val) + [1]*len(X_out))

def auc_scorer(est, X, y):
    return roc_auc_score(y, -est.decision_function(X))

result = permutation_importance(iso, X_perm, y_perm, scoring=auc_scorer, n_repeats=30, random_state=0)
importance = pd.DataFrame({"feature": FEATURES, "importance_mean": result.importances_mean,
                           "importance_std": result.importances_std}).sort_values("importance_mean", ascending=False)
print("Isolation Forest permutation importance (validation+outlier, AUC 기준):")
print(importance.round(4).to_string(index=False))

Isolation Forest permutation importance (validation+outlier, AUC 기준):
            feature  importance_mean  importance_std
normal_center_nrmse           0.0119          0.0039
            ai0_rms           0.0008          0.0009
      cur_ai1_resid           0.0006          0.0010
      cur_ai0_resid           0.0005          0.0010
      ratio_ai1_ai0          -0.0000          0.0006
            ai1_rms          -0.0002          0.0004


## 7-1. One-Class SVM 설명력

RBF 커널은 비선형이라 Mahalanobis처럼 점수를 피처별로 정확히 더해 분해할 수 없다.
(1) Isolation Forest와 같은 방식의 permutation importance(전역), (2) KernelSHAP(개별
세그먼트별)을 쓴다. 국소 유한차분(기울기)도 먼저 시도했으나, 확신 있게 분류된 지점은
RBF 커널값이 거의 0으로 포화돼 기울기도 0이 되어 설명이 안 됐다 -- SHAP은 기울기가 아니라
배경표본과의 실제 함수값 비교(좌표 섭동)를 쓰므로 포화 지점에서도 분해가 가능하다.

In [ ]:
def auc_scorer(est, X, y):
    return roc_auc_score(y, ocsvm_score(X))

r_svm = permutation_importance(ocsvm, X_perm, y_perm, scoring=auc_scorer, n_repeats=30, random_state=0)
importance_svm = pd.DataFrame({"feature": FEATURES, "importance_mean": r_svm.importances_mean,
                               "importance_std": r_svm.importances_std}).sort_values("importance_mean", ascending=False)
print("One-Class SVM permutation importance:")
print(importance_svm.round(4).to_string(index=False))

One-Class SVM permutation importance:
            feature  importance_mean  importance_std
normal_center_nrmse           0.1001          0.0097
      cur_ai0_resid           0.0387          0.0051
            ai0_rms           0.0282          0.0048
            ai1_rms           0.0000          0.0000
      cur_ai1_resid           0.0000          0.0000
      ratio_ai1_ai0           0.0000          0.0000


In [ ]:
import shap

background = shap.kmeans(X_train, 20)
explainer = shap.KernelExplainer(ocsvm_score, background)

shap_values = explainer.shap_values(X_out, nsamples=100)
print(f"base_value(배경 평균 점수) = {explainer.expected_value:.3f}")
rows = []
for sid, score, row in zip(ot.loc[ovalid, 'seg_id'], ocsvm_score(X_out), shap_values):
    top = np.argsort(-np.abs(row))[:2]
    rows.append({"seg_id": sid, "score": round(float(score), 3),
                 "top1": f"{FEATURES[top[0]]}={row[top[0]]:.3f}",
                 "top2": f"{FEATURES[top[1]]}={row[top[1]]:.3f}"})
print(pd.DataFrame(rows).to_string(index=False))

print()
print("경계 근처(04 오경보 하드케이스, normal seg523·529·585)에서의 SHAP:")
hard = TABLE['normal'].query("split=='test'")
hard = hard[hard.seg_id.isin([523,529,585])][FEATURES].to_numpy(float)
shap_hard = explainer.shap_values(hard, nsamples=100)
for score, row in zip(ocsvm_score(hard), shap_hard):
    top = np.argsort(-np.abs(row))[:3]
    print(f"  score={score:.4f}: " + ", ".join(f"{FEATURES[j]}={row[j]:.4f}" for j in top))

  0%|          | 0/17 [00:00<?, ?it/s]

base_value(배경 평균 점수) = -0.659
 seg_id  score                      top1                      top2
      1  4.665       cur_ai0_resid=1.122 normal_center_nrmse=1.122
      2  4.665       cur_ai0_resid=1.078             ai0_rms=1.078
      3  4.665 normal_center_nrmse=4.019             ai0_rms=0.360
      4  4.665             ai0_rms=1.059 normal_center_nrmse=1.059
      6  4.665             ai0_rms=1.106 normal_center_nrmse=1.106
      7  4.665 normal_center_nrmse=1.221       cur_ai0_resid=1.221
      8  4.665       cur_ai0_resid=1.127 normal_center_nrmse=1.127
     11  4.665       cur_ai0_resid=1.162 normal_center_nrmse=1.162
     12  4.665             ai0_rms=1.146       cur_ai0_resid=1.146
     13  4.665 normal_center_nrmse=1.042             ai0_rms=1.042
     14  4.665 normal_center_nrmse=1.278       cur_ai0_resid=1.278
     15  4.665 normal_center_nrmse=1.468       cur_ai0_resid=1.468
     16  4.665 normal_center_nrmse=1.212             ai0_rms=1.211
     17  4.665 normal_center_nrm

  0%|          | 0/3 [00:00<?, ?it/s]

  score=0.5652: normal_center_nrmse=0.8561, ratio_ai1_ai0=0.2046, ai0_rms=0.1502
  score=1.9944: normal_center_nrmse=1.3108, ai0_rms=0.6348, cur_ai0_resid=0.4981
  score=1.3091: normal_center_nrmse=1.4757, cur_ai0_resid=0.2642, ratio_ai1_ai0=0.1760


**결과:** permutation importance는 `normal_center_nrmse`(0.100)가 압도적이고 `cur_ai0_resid`(0.039)·`ai0_rms`(0.028)가 보조, 나머지 3개는 0 -- Mahalanobis·Isolation Forest와 같은 결론이다.

KernelSHAP은 모든 outlier(점수가 동일하게 포화된 17개 전부)에서 분해가 가능했다. **대부분의 "시끄러움" 유형(12개)은 `normal_center_nrmse`와 `ai0_rms`/`cur_ai0_resid`가 거의 동률로 공동 1위**다 -- 여러 피처가 같이 쏠린다는 뜻. 반면 **seg19·20은 2위가 `cur_ai1_resid`로 바뀐다** -- Mahalanobis의 기여도 분해(7절)와 정확히 같은 결론이다. 경계 근처(normal 하드케이스)에서도 `normal_center_nrmse`가 1위로 일관된다.

**세 가지 설명력 방법(Mahalanobis 분해, Isolation Forest permutation importance, OCSVM SHAP)이 전부 같은 결론으로 수렴한다**: `normal_center_nrmse`가 압도적 주 피처이고, `cur_ai1_resid`가 seg19·20이라는 특정 서브유형에만 추가로 기여한다. 모델이 달라도 피처가 하는 역할은 일관되게 나타나, 03~05에서 쌓아온 피처 해석(02 feature_candidates.md)이 모델 선택과 무관하게 견고하다는 근거가 된다.

## 해석 시 주의

04와 동일하다 — 정상 test 오경보 0건이 실제 운영 오경보율 0을 보장하지 않으며, normal/outlier의
날짜·운전조건 교락도 그대로 남아 있다. Mahalanobis·Isolation Forest 모두 피처 선정에 이미
outlier를 참고했다는 한계를 공유한다.

## 실행 결과 요약

**최종 피처**: `cur_ai0_resid, cur_ai1_resid, normal_center_nrmse` (3개) — 공선성
기반 단계적 VIF 제거로 6개→4개→3개까지 축소(성능과 무관한 구조적 기준; 근거는
`docs/02_feature_candidates.md`). **채택 모델**: Mahalanobis.

### 단일 분할(validation/test, n≥10 기준)

| 모델 | split | 정상탐지율 | 오경보율 | 이상탐지율 | BA |
|---|---|---:|---:|---:|---:|
| Mahalanobis | test | 100% | 0%(0/109) | 100%(17/17) | 100% |
| Isolation Forest | test | 100% | 0%(0/109) | 76.47%(13/17) | 88.24% |
| One-Class SVM | test | 100% | 0%(0/109) | 100%(17/17) | 100% |
| (참고, 04) NRMSE 단독 | test | 97.25% | 2.75%(3/109) | 100%(17/17) | 98.62% |
| (참고, 04) 6개 피처 OR | test | 80.73% | 19.27%(21/109) | 100% | 90.37% |

Mahalanobis·One-Class SVM은 완벽(FP 0%/TP 100%), Isolation Forest만 seg3·19·20("조용함"
실패유형: 전류·진동 각자는 정상, 전류 대비 진동 잔차만 이상)을 놓친다. 04의 "6개 피처
OR"이 나빴던 건 피처가 아니라 결합 방식(OR)이 거칠었기 때문이라는 가설이 확인된다.

### 블록 k-fold 검증 — 세 모델 공정 비교 (## 6-1)

| 모델 | FP 평균 | FP 95% CI | TP 평균 | TP sd |
|---|---:|---:|---:|---:|
| Mahalanobis | 0.19% | [0.00%, 0.55%] | **100%** | 0 |
| Isolation Forest | 0.19% | [0.00%, 0.55%] | 95.1% | 2.4%p |
| One-Class SVM | 0.37% | [0.00%, 1.11%] | **100%** | 0 |

FP는 세 모델 다 낮고 신뢰구간이 겹친다. **Isolation Forest의 fold 평균 TP(95.1%)가 단일
분할 test TP(76.47%)보다 훨씬 높다** — 단일 분할에서의 저조함은 그 특정 train 샘플의
우연이었다는 뜻이고, 모델 하나만 또는 단일 분할만 보고하면 오판할 수 있다는 직접적인
근거다. **결론: Mahalanobis가 TP 100%/sd 0·FP 0.19%로 최상이고, One-Class SVM이 TP는
동일하게 완벽해 사실상 공동 1위다. Isolation Forest는 fold 평균으로도 가장 불안정하다.**

### 설명력 (## 7, 7-1)

Mahalanobis D² 분해·Isolation Forest permutation importance·One-Class SVM
permutation importance+KernelSHAP **세 방법 모두 `normal_center_nrmse`를 압도적 1위로
꼽아 일관된다.** `cur_ai1_resid`는 seg19·20("조용함" 유형)에서만 2위로 올라온다 — 이
피처가 설계 의도(전류 대비 진동 잔차)대로 정확히 그 서브유형을 잡아준다는 뜻이다.
국소 유한차분(기울기) 방식은 확신 있는 이상 지점에서 RBF 커널이 포화돼 실패했고, SHAP
(좌표 섭동 기반)으로 대체해 해결했다(`shap` 패키지를 my_base에 추가 설치, README 반영).

### 다음 단계

Mahalanobis를 기준 모델로 채택한다. validation은 재조정하지 않고 그대로 뒀으므로, 이후
N중M 지속성 규칙을 validation에서 설정할 때 입력으로 쓸 수 있다. `03_modeling_strategy.md`
"평가 보강 사항" 중 전환구간(00:55~01:17) 점수의 시간축 플롯 진단은 아직 미적용 —
모델이 고정된 지금 다음으로 진행한다.